In [ ]:
import pandas as pd
import glob
import os
import chardet

data_path = r"/content/ST data"  # папка с твоими файлами
files = glob.glob(os.path.join(data_path, "*.csv"))

all_data = []

for file in files:
    # --- определяем кодировку ---
    with open(file, "rb") as f:
        rawdata = f.read(50000)
        result = chardet.detect(rawdata)
    encoding = result["encoding"]
    print(f"Файл {os.path.basename(file)} кодировка: {encoding}")

    try:
        # читаем с табуляцией
        df = pd.read_csv(file, sep="\t", encoding=encoding)
        # нормализуем заголовки
        df.columns = [c.strip().lower() for c in df.columns]
        all_data.append(df)
        print(f"Файл {os.path.basename(file)} успешно загружен, строк: {len(df)}")
    except Exception as e:
        print(f"Ошибка при чтении {file}: {e}")

# --- объединяем ---
if not all_data:
    raise ValueError("Нет успешно загруженных файлов!")

df_all = pd.concat(all_data, ignore_index=True)

# --- находим колонку app id ---
app_id_col = next((c for c in df_all.columns if "app id" in c), None)
if not app_id_col:
    raise ValueError(f"Колонка App ID не найдена! Есть колонки: {df_all.columns.tolist()}")

# --- удаляем дубликаты по app id ---
df_all = df_all.drop_duplicates(subset=[app_id_col], keep="first")

# --- сохраняем итог ---
output_file = os.path.join(data_path, "all_apps_unique.xlsx")
df_all.to_excel(output_file, index=False)

print(f"\nГотово! Итоговый файл сохранён: {output_file}")
print(f"Уникальных приложений: {len(df_all)}")


**Подготовка мета файла Downloads**

In [ ]:
import pandas as pd
import glob
import os

# === Путь к папке ===
path = r"C:\Users\vyvys\anaconda_projects\Merging folder"
files = glob.glob(os.path.join(path, "Google Play Download Source by Absolute Downloads*.csv"))

if not files:
    raise FileNotFoundError("Не найдено файлов с нужным паттерном")

all_data = []

for file in files:
    print(f"Обрабатываю: {os.path.basename(file)}")
    df = pd.read_csv(file, sep=',', thousands=',')

    # melt в длинный формат
    df_long = df.melt(id_vars='Date', var_name='app_col', value_name='Downloads')

    # разбираем тип трафика
    df_long['Traffic Source'] = df_long['app_col'].str.extract(
        r'(Organic Search|Organic Browse|Paid Ads|Paid Search|Web Browser)'
    )

    # разбираем имя приложения
    df_long['App Name'] = df_long['app_col'].str.extract(
        r'^(.*?) (?:Organic Search|Organic Browse|Paid Ads|Paid Search|Web Browser) Downloads'
    )

    df_long = df_long[['Date', 'App Name', 'Traffic Source', 'Downloads']]
    df_long['Date'] = pd.to_datetime(df_long['Date'], errors='coerce')
    df_long['Downloads'] = df_long['Downloads'].fillna(0).round().astype('Int64')

    all_data.append(df_long)

# === Объединяем всё ===
merged_df = pd.concat(all_data, ignore_index=True)

# === Пивотим в широкий формат ===
df_pivot = merged_df.pivot_table(
    index=['Date', 'App Name'],
    columns='Traffic Source',
    values='Downloads',
    aggfunc='sum',
    fill_value=0
).reset_index()

# --- Убираем имя уровня колонок ---
df_pivot.columns.name = None

# --- Обязательно добавим отсутствующие колонки ---
for col in ["Organic Search", "Organic Browse", "Paid Ads", "Paid Search", "Web Browser"]:
    if col not in df_pivot.columns:
        df_pivot[col] = 0

# --- Финальный порядок ---
cols = ['Date', 'App Name', 'Organic Search', 'Organic Browse', 'Paid Ads', 'Paid Search', 'Web Browser']
df_pivot = df_pivot[cols]

# === Сохраняем результат ===
output_path = os.path.join(path, "GooglePlay_DownloadSources_Appwise.csv")
df_pivot.to_csv(output_path, index=False)

print(f"✅ Файл сохранён: {output_path}, размер: {df_pivot.shape}")
df_pivot.head()


**Подготовка мета файла Category**

In [ ]:
import pandas as pd
import glob
import os

# === Путь к папке ===
path = r"C:\Users\vyvys\anaconda_projects\Merging folder"
files = glob.glob(os.path.join(path, "Google Play Category Rankings*.csv"))

if not files:
    raise FileNotFoundError("Не найдено файлов с нужным паттерном")

all_data = []

for file in files:
    print(f"Обрабатываю: {os.path.basename(file)}")

    # Читаем файл с пропуском первой строки
    df = pd.read_csv(file, encoding="utf-16", sep="\t", skiprows=1)

    # Получаем категорию из первой строки данных (она хранится в колонке 'topselling_free')
    category = df['topselling_free'].iloc[0]

    # Удаляем эту строку, она дублирует заголовки
    df = df[1:].copy()

    # Заполняем колонку Category вручную
    df['Category'] = category

    all_data.append(df)

# Объединяем
final_df = pd.concat(all_data, ignore_index=True)

# Приводим дату
final_df['Date'] = pd.to_datetime(final_df['Date'], dayfirst=True, errors='coerce')

# Сохраняем
output_path = os.path.join(path, "GooglePlay_CategoryRankings_Merged.csv")
final_df.to_csv(output_path, index=False, encoding="utf-8")
print(f"Файл сохранён: {output_path}, размер: {final_df.shape}")

final_df.head()


In [ ]:
import pandas as pd
import glob
import os

# === Путь к папке ===
path = r"C:\Users\vyvys\anaconda_projects\Merging folder"
files = glob.glob(os.path.join(path, "Google Play DAU*.csv"))

if not files:
    raise FileNotFoundError("Не найдено файлов с паттерном 'Google Play DAU*.csv'")

all_data = []

for file in files:
    print(f"Обрабатываю: {os.path.basename(file)}")
    # Читаем с учётом кодировки и табуляции
    df = pd.read_csv(file, encoding="utf-16", sep="\t")
    all_data.append(df)

# === объединяем ===
final_df = pd.concat(all_data, ignore_index=True)

# === дата ===
final_df['Date'] = pd.to_datetime(final_df['Date'], errors='coerce')

# === сохраняем ===
output_path = os.path.join(path, "GooglePlay_DAU_Merged.csv")
final_df.to_csv(output_path, index=False, encoding="utf-8")
print(f"Файл сохранён: {output_path}, размер: {final_df.shape}")

final_df.head()


In [ ]:
import pandas as pd
import glob
import os

# === Путь к папке ===
path = r"C:\Users\vyvys\anaconda_projects\Merging folder"
files = glob.glob(os.path.join(path, "Google Play ARPDAU*.csv"))

if not files:
    raise FileNotFoundError("Не найдено файлов с паттерном 'Google Play ARPDAU*.csv'")

all_data = []

for file in files:
    print(f"Обрабатываю: {os.path.basename(file)}")
    # Читаем с учётом кодировки и табуляции
    df = pd.read_csv(file, encoding="utf-16", sep="\t")
    all_data.append(df)

# === объединяем ===
final_df = pd.concat(all_data, ignore_index=True)

# === дата ===
final_df['Date'] = pd.to_datetime(final_df['Date'], errors='coerce')

# === сохраняем ===
output_path = os.path.join(path, "GooglePlay_ARPDAU_Merged.csv")
final_df.to_csv(output_path, index=False, encoding="utf-8")
print(f"Файл сохранён: {output_path}, размер: {final_df.shape}")

final_df.head()


In [ ]:
import pandas as pd
from functools import reduce

# === читаем файл с загрузками ===
df_downloads = pd.read_csv(
    'C:/Users/vyvys/anaconda_projects/Merging folder/GooglePlay_DownloadSources_cleaned.csv'
)

# Переименуем колонку с app id в общий формат
df_downloads.rename(columns={"app id": "App ID", "app name": "App Name"}, inplace=True)

# === читаем остальные файлы ===
df_category = pd.read_csv(
    'C:/Users/vyvys/anaconda_projects/Merging folder/GooglePlay_CategoryRankings_Merged.csv'
)
df_arpdau = pd.read_csv(
    'C:/Users/vyvys/anaconda_projects/Merging folder/GooglePlay_ARPDAU_Merged.csv'
)
df_dau = pd.read_csv(
    'C:/Users/vyvys/anaconda_projects/Merging folder/GooglePlay_DAU_Merged.csv'
)

# === убираем строки без App ID ===
df_downloads = df_downloads.dropna(subset=["App ID"])
df_downloads = df_downloads[df_downloads["App ID"].str.strip() != ""]

# === убираем App Name, чтобы не мешал при merge ===
for df in [df_downloads, df_category, df_arpdau, df_dau]:
    df.drop(columns=["App Name"], errors="ignore", inplace=True)

# === объединение всех датасетов (outer join) ===
dfs = [df_downloads, df_category, df_arpdau, df_dau]
df_final = reduce(lambda left, right: pd.merge(left, right, on=["App ID", "Date"], how="outer"), dfs)

# === удаляем дубликаты только по App ID и Date ===
df_final = df_final.drop_duplicates(subset=["App ID", "Date"])

# === информация о результате ===
print("Итоговый размер:", df_final.shape)
print(df_final.head())

# === сохраняем ===
df_final.to_csv(
    "C:/Users/vyvys/anaconda_projects/Merging folder/GooglePlay_AllMerged.csv", index=False
)
print("Файл сохранён: GooglePlay_AllMerged.csv")


In [ ]:
df_final = df_final.drop_duplicates()
summary = pd.DataFrame({
    'non_null_count': df_final.count(),
    'null_count': df_final.isnull().sum(),
    'dtype': df_final.dtypes
})
print(summary)


In [ ]:
import pandas as pd

# === пути ===
p_downloads = r"C:/Users/vyvys/anaconda_projects/Merging folder/GooglePlay_DownloadSources_cleaned.csv"
p_category  = r"C:/Users/vyvys/anaconda_projects/Merging folder/GooglePlay_CategoryRankings_Merged.csv"
p_arpdau    = r"C:/Users/vyvys/anaconda_projects/Merging folder/GooglePlay_ARPDAU_Merged.csv"
p_dau       = r"C:/Users/vyvys/anaconda_projects/Merging folder/GooglePlay_DAU_Merged.csv"

def load_norm(path):
    df = pd.read_csv(path)
    # унифицируем имена
    df = df.rename(columns={"app id":"App ID", "app name":"App Name"}, errors="ignore")
    # ключи
    if "App ID" in df.columns:
        df["App ID"] = df["App ID"].astype(str).str.strip().str.lower()
    if "Date" in df.columns:
        df["Date"] = pd.to_datetime(df["Date"], errors="coerce").dt.normalize()
    # лишнее имя
    df.drop(columns=["App Name"], errors="ignore", inplace=True)
    # чистим пустые ключи
    df = df.dropna(subset=["App ID","Date"])
    df = df[df["App ID"].str.strip() != ""]
    return df

df_downloads = load_norm(p_downloads)
df_category  = load_norm(p_category)
df_arpdau    = load_norm(p_arpdau)
df_dau       = load_norm(p_dau)

# === БАЗА: DOWNLOADS ===
base = df_downloads.copy()

def report_match(before_df, after_df, new_cols, label):
    """Печать процента строк базы, где 'что-то' заматчилось из новых колонок."""
    mask_any = after_df[new_cols].notna().any(axis=1)
    matched = mask_any.sum()
    print(f"{label}: matched rows in base = {matched:,} / {len(before_df):,} "
          f"({matched/len(before_df)*100:.2f}%)")

# === LEFT JOIN: ARPDAU ===
# (В этом файле: Downloads, Revenue ($), RPD ($), ARPDAU ($))
arp_cols = ["Downloads","Revenue ($)","RPD ($)","ARPDAU ($)"]
df_arpdau = df_arpdau[["App ID","Date"] + [c for c in arp_cols if c in df_arpdau.columns]]
merged = base.merge(df_arpdau, on=["App ID","Date"], how="left")
report_match(base, merged, [c for c in arp_cols if c in merged.columns], "ARPDAU join")
base = merged

# === LEFT JOIN: DAU ===
dau_cols = ["DAU"]
df_dau = df_dau[["App ID","Date"] + [c for c in dau_cols if c in df_dau.columns]]
merged = base.merge(df_dau, on=["App ID","Date"], how="left")
report_match(base, merged, [c for c in dau_cols if c in merged.columns], "DAU join")
base = merged

# === LEFT JOIN: CATEGORY/RANK ===
cat_cols = ["Rank","Updated","Category","WasRanked"]
df_category = df_category[["App ID","Date"] + [c for c in cat_cols if c in df_category.columns]]
merged = base.merge(df_category, on=["App ID","Date"], how="left")
report_match(base, merged, [c for c in cat_cols if c in merged.columns], "CATEGORY join")
base = merged

# === (опционально) аккуратно заполним категорию вперёд/назад в рамках App ID ===
# Это НЕ выдумывает Rank, а только тянет Category/Updated, если она стабильна между датами
fill_category = True
if fill_category and "Category" in base.columns:
    base = base.sort_values(["App ID","Date"])
    base["Category"] = base.groupby("App ID")["Category"].ffill().bfill()
    if "Updated" in base.columns:
        base["Updated"] = base.groupby("App ID")["Updated"].ffill().bfill()

# === срез по дате от 2023-01-01 (если нужно) ===
base_2023 = base[base["Date"] >= pd.Timestamp("2023-01-01")].copy()

# === финальная диагностика ===
print("\n--- FINAL SHAPE ---")
print("All rows:", len(base), " | Unique App ID:", base["App ID"].nunique())
print("2023+ rows:", len(base_2023), " | Unique App ID:", base_2023["App ID"].nunique())

# доля пропусков по ключевым метрикам после джоинов
check_cols = [
    "Organic Search","Organic Browse","Paid Ads","Paid Search","Web Browser",
    "Downloads","Revenue ($)","RPD ($)","ARPDAU ($)","DAU","Rank","Category"
]
check_cols = [c for c in check_cols if c in base_2023.columns]
miss_share = base_2023[check_cols].isna().mean().sort_values(ascending=False)
print("\nMissing share (2023+):")
print((miss_share*100).round(2).astype(str) + "%")

# === сохранить ===
out_full  = r"C:/Users/vyvys/anaconda_projects/Merging folder/GooglePlay_AllMerged_LEFT.csv"
out_2023  = r"C:/Users/vyvys/anaconda_projects/Merging folder/GooglePlay_AllMerged_LEFT_2023.csv"
base.to_csv(out_full, index=False)
base_2023.to_csv(out_2023, index=False)
print(f"\nSaved:\n- {out_full}\n- {out_2023}")


In [ ]:
import pandas as pd
import numpy as np

# === вход/выход ===
in_path  = r"C:/Users/vyvys/anaconda_projects/Merging folder/GooglePlay_AllMerged_LEFT.csv"
out_path = r"C:/Users/vyvys/anaconda_projects/Merging folder/GooglePlay_AllMerged_FINAL_2021.csv"

df = pd.read_csv(in_path)
df["Date"] = pd.to_datetime(df["Date"], errors="coerce").dt.normalize()
df["App ID"] = df["App ID"].astype(str).str.strip().str.lower()
df = df.sort_values(["App ID","Date"])

# 1) Удаляем столбцы-имена и Downloads
cols_to_drop = {"name_short","name clean","name_clean","name short","Name Short","Downloads","App Name"}
df.drop(columns=[c for c in cols_to_drop if c in df.columns], inplace=True, errors="ignore")

# === helpers ===
def fill_rev_like_group(metric: pd.Series, dau: pd.Series, global_median: float) -> pd.Series:
    """
    Заполнение для Revenue/RPD/ARPDAU по правилу:
    - если DAU == 0 и metric NaN -> 0
    - иначе: rolling median(7, center) -> median по AppID -> глобальная медиана
    """
    s = metric.copy()

    if dau is not None and "Int64" in str(dau.dtype):
        dau_vals = dau.astype("float").values
    else:
        dau_vals = dau.values if dau is not None else None

    # 0 на краях/внутри, если DAU==0 и метрика NaN
    if dau_vals is not None:
        mask_zero_dau_and_nan = (pd.Series(dau_vals, index=s.index).fillna(0) == 0) & s.isna()
        s.loc[mask_zero_dau_and_nan] = 0

    # дальше — мягкая импутация
    if s.notna().any():
        roll_med = s.rolling(window=7, min_periods=1, center=True).median()
        s = s.fillna(roll_med)
        s = s.fillna(s.median())

    # крайний случай
    if s.isna().any():
        s = s.fillna(global_median)

    return s

def fill_dau_group(s: pd.Series) -> pd.Series:
    """DAU: нули на краях, внутри — rolling median(7) -> медиана по App ID -> 0"""
    if s.notna().sum() == 0:
        return s.fillna(0).astype("Int64")
    notna = s.notna()
    leading_na = ~notna.cummax()
    trailing_na = (~notna[::-1].cummax())[::-1]
    edge_na = leading_na | trailing_na
    mid_na = s.isna() & ~edge_na

    s.loc[edge_na] = 0
    if mid_na.any():
        roll_med = s.rolling(window=7, min_periods=1, center=True).median()
        s.loc[mid_na] = roll_med.loc[mid_na]
    if s.isna().any():
        s = s.fillna(s.median())
    s = s.fillna(0)
    try:
        s = s.round().astype("Int64")
    except Exception:
        pass
    return s

# 2) DAU
if "DAU" in df.columns:
    df["DAU"] = df.groupby("App ID", group_keys=False)["DAU"].apply(fill_dau_group)
else:
    # если вдруг нет DAU, создадим пустую для логики ниже
    df["DAU"] = pd.Series([pd.NA]*len(df), dtype="Int64")

# 3) Revenue / RPD / ARPDAU
rev_like_cols = [c for c in ["Revenue ($)","RPD ($)","ARPDAU ($)"] if c in df.columns]
global_meds = {c: df[c].median() for c in rev_like_cols}

for col in rev_like_cols:
    df[col] = df.groupby("App ID", group_keys=False).apply(
        lambda g: fill_rev_like_group(g[col], g["DAU"], global_meds[col])
    ).reset_index(level=0, drop=True)

# 4) WasRanked: 1, если когда-либо Rank был не NaN для App ID
if "Rank" in df.columns:
    has_rank = df.groupby("App ID")["Rank"].apply(lambda s: int(s.notna().any()))
    df = df.merge(has_rank.rename("WasRanked_new"), on="App ID", how="left")
    df["WasRanked"] = df["WasRanked_new"].fillna(0).astype("int8")
    df.drop(columns=["WasRanked_new"], inplace=True, errors="ignore")

# 5) небольшой отчёт
report_cols = [c for c in ["Revenue ($)","RPD ($)","ARPDAU ($)","DAU","Rank","Category","WasRanked"] if c in df.columns]
miss = df[report_cols].isna().mean().sort_values(ascending=False)
print("Missing share after fill:\n", (miss*100).round(2).astype(str) + "%")

# 6) сохранить
df.to_csv(out_path, index=False)
print("Saved:", out_path)


In [ ]:
# prepare_dataset_add_columns.py
# -*- coding: utf-8 -*-
import pandas as pd
from pathlib import Path

INPUT = Path("C:/Users/vyvys/anaconda_projects/Merging folder/GooglePlay_AllMerged_FINAL_2021.csv")
OUTPUT = INPUT.with_name(INPUT.stem + "_augmented.csv")

# Загружаем
df = pd.read_csv(INPUT)

# Приводим имена к точным, как у тебя
rename_map = {
    "Date": "Date",
    "Organic Search": "Organic Search",
    "Organic Browse": "Organic Browse",
    "Paid Ads": "Paid Ads",
    "Paid Search": "Paid Search",
    "Web Browser": "Web Browser",
    "App ID": "App ID",
    "Revenue ($)": "Revenue ($)",
    "RPD ($)": "RPD ($)",
    "ARPDAU ($)": "ARPDAU ($)",
    "DAU": "DAU",
    "Rank": "Rank",
    "Updated": "Updated",
    "Category": "Category",
    "WasRanked": "WasRanked",
}
df = df.rename(columns=rename_map)

# Новые столбцы:
# 1) Organic Traffic = Organic Search + Organic Browse
for col in ["Organic Search", "Organic Browse", "Paid Ads", "Paid Search", "Web Browser"]:
    if col in df.columns:
        df[col] = df[col].fillna(0)

df["Organic Traffic"] = df["Organic Search"] + df["Organic Browse"]

# 2) Paid Traffic = Paid Ads + Paid Search + Web Browser
df["Paid Traffic"] = df["Paid Ads"] + df["Paid Search"] + df["Web Browser"]

# Сохраняем
df.to_csv(OUTPUT, index=False)
print(f"Saved: {OUTPUT}")
